In [1]:
from Lists.abbreviations import INTERNET_SLANG
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from Lists.emoticons import EMOTICONS
from spellchecker import SpellChecker
from Lists.emojis import EMO_UNICODE
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import pandas as pd
import contractions
import unicodedata
import nltk
import re

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never'}
lemmatizer = WordNetLemmatizer()
spell = SpellChecker()

tqdm.pandas()

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df = pd.read_csv('./OLID/OLID.csv')

In [3]:
df.head()

,text,label_A,label_B,label_C
0,Has @USER quit? I've not heard of any #knifecr...,NOT,NaN,NaN
1,"In celebration of Emancipation Day, we urge yo...",NOT,NaN,NaN
2,@USER @USER It’d be a literal dream come true ...,NOT,NaN,NaN
3,Brilliant news to read that Hoggy has signed a...,NOT,NaN,NaN
4,@USER She speaks of the truth 😌,NOT,NaN,NaN


In [4]:
df = df[~((df['label_A'] == 'OFF') & (df['label_B'] == 'TIN') & (df['label_C'].isna()))]

label_dictionary = {
    ('NOT', None, None): 'Not Offensive',
    ('OFF', 'UNT', None): 'Untargeted Offense',
    ('OFF', 'TIN', 'IND'): 'Targeted Offense - Individual',
    ('OFF', 'TIN', 'GRP'): 'Targeted Offense - Group',
    ('OFF', 'TIN', 'OTH'): 'Targeted Offense - Other'
}


def map_type(row):
    key = (row['label_A'], row['label_B'], row['label_C'])
    return label_dictionary.get(key, "Unknown")

df[['label_A', 'label_B', 'label_C']] = df[['label_A', 'label_B', 'label_C']].replace({pd.NA: None, 'nan': None, 'None': None})

df['type'] = df.apply(map_type, axis=1)

In [5]:
type_labels = {label: idx for idx, label in enumerate(df['label_A'].unique())}
print(type_labels)
df['label'] = df['label_A'].map(type_labels)

{'NOT': 0, 'OFF': 1}


In [6]:
df.head()

,text,label_A,label_B,label_C,type,label
0,Has @USER quit? I've not heard of any #knifecr...,NOT,None,None,Not Offensive,0
1,"In celebration of Emancipation Day, we urge yo...",NOT,None,None,Not Offensive,0
2,@USER @USER It’d be a literal dream come true ...,NOT,None,None,Not Offensive,0
3,Brilliant news to read that Hoggy has signed a...,NOT,None,None,Not Offensive,0
4,@USER She speaks of the truth 😌,NOT,None,None,Not Offensive,0


In [7]:
counts = df['type'].value_counts(dropna=False)
print("Numărul de exemple pentru fiecare label:")
for label, count in counts.items():
    print(f"{label}: {count}")

Numărul de exemple pentru fiecare label:
Not Offensive: 2991
Untargeted Offense: 1452
Targeted Offense - Individual: 1057
Targeted Offense - Group: 349
Targeted Offense - Other: 140


In [8]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}

def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text

def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text

def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text

def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"  # miscellaneous symbols
        u"\U0001F900-\U0001F9FF"  # supplemental symbols
        u"\U0001F200-\U0001F251"  # enclosed characters
        u"\U0001F004-\U0001F0CF"  # playing cards
        u"\U00002B50"  # star symbol
        "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text

def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [9]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [10]:
 def correction_stopwords_lemmatize(text):
        if not isinstance(text, str):
            return ""

        text = contractions.fix(text)
        tokens = word_tokenize(text)
        corrected_and_lemmatized_tokens = []

        for word in tokens:
            if word in stop_words:
                continue

            corrected = word
            if spell.unknown([word]):
                possible_correction = spell.correction(word)
                if possible_correction is not None:
                    corrected = possible_correction

            lemmatized = lemmatizer.lemmatize(corrected)
            corrected_and_lemmatized_tokens.append(lemmatized)

        return " ".join(corrected_and_lemmatized_tokens)

In [11]:
sequencePattern = r"(.)\1\1+"
seqReplacePattern = r"\1\1"

In [12]:
def preprocess_text(df_input):
    df_preprocessing=df_input.copy()
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r"http\S+", "URL", regex=True)  # Remove URLs
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # Remove user mentions
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # Remove hashtags
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(sequencePattern, seqReplacePattern, regex=True)  # Remove repetition

    df_preprocessing['text'] = df_preprocessing['text'].progress_apply(lambda x: convert_emojis_and_slang(x))
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\d+', '', regex=True)  # Remove digits
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'[^\w\s]', '', regex=True)  # Remove punctuation
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\s+', ' ', regex=True)  # Remove spaces
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'[^a-zA-Z\s]', '', regex=True)  # Only words
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\s+', ' ', regex=True).str.strip()  # Trim spaces
    df_preprocessing['text'] = df_preprocessing['text'].str.lower()  # Convert text to lowercase
    df_preprocessing['text'] = df_preprocessing['text'].progress_apply(lambda x: correction_stopwords_lemmatize(x))

    df_preprocessing = df_preprocessing.drop_duplicates()

    return df_preprocessing

In [13]:
df = preprocess_text(df)

  0%|          | 0/5989 [00:00<?, ?it/s]

  0%|          | 0/5989 [00:00<?, ?it/s]

In [14]:
df.head()

,text,label_A,label_B,label_C,type,label
0,user quit not heard knifecrime today,NOT,None,None,Not Offensive,0
1,celebration emancipation day urge emancipate r...,NOT,None,None,Not Offensive,0
2,user user would literal dream come true win es...,NOT,None,None,Not Offensive,0
3,brilliant news read doggy signed new contract ...,NOT,None,None,Not Offensive,0
4,user speaks truth relievedface,NOT,None,None,Not Offensive,0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5780 entries, 0 to 5992
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     5780 non-null   object
 1   label_A  5780 non-null   object
 2   label_B  2885 non-null   object
 3   label_C  1456 non-null   object
 4   type     5780 non-null   object
 5   label    5780 non-null   int64 
dtypes: int64(1), object(5)
memory usage: 316.1+ KB


In [16]:
df.to_csv('./OLID_classifiers.csv',index=False)